# Tracking and PID efficiencies
## Do a weighted mean to find the systematic uncertainty due to tracking and PID

### Pions or kaons?

In [1]:
//const int ID = 321; // Kaons
const int ID = 211; // Pions

### What efficiencies? (In percent)

In [2]:
// Kaon tracking efficiencies
/*std::vector<double> EffDiff{
    3.77,
    -3.96,
    0.48,
    0.48,
    0.31,
    0.03,
    -0.03,
    -0.26,
    -0.21,
    -0.09
};*/

In [3]:
// Kaon PID efficiencies
/*std::vector<double> EffDiff{
    0.0,
    1.08,
    0.35,
    0.0,
    0.17,
    0.06,
    0.07,
    -0.06,
    -0.07,
    -0.05
};*/

In [4]:
// Pion tracking efficiencies
/*std::vector<double> EffDiff{
    -3.51,
    0.23,
    -0.80,
    -0.66,
    -0.31,
    0.04,
    -0.08,
    -0.14,
    -0.06,
    -0.04
};*/

In [5]:
// Pion PID efficiencies
std::vector<double> EffDiff{
    -1.61,
    -0.15,
    -0.05,
    -0.09,
    -0.09,
    -0.22,
    -0.32,
    -0.63,
    -0.88,
    -0.77
};

### Histogram of momenta

In [6]:
std::vector<std::size_t> Hist(EffDiff.size());

### Function to find bin

In [7]:
std::size_t FindBin(double Momentum) {
    if(Momentum < 0.0) {
        std::cout << "Momentum cannot be negative!\n";
        return -1;
    } else if(Momentum > 0.9) {
        return 9;
    } else {
        return static_cast<std::size_t>(Momentum*10.0);
    }
}

### Load MC truth tuple

In [8]:
std::string TruthTupleFilename("/data/bes3/tat/KKpipi_StrongPhase_Analysis_4Bins/");
TruthTupleFilename += "TruthTuples/BinnedTruthTuples/Kpi/KKpipi_vs_Kpi_TruthTuple_Binned.root";
TChain TruthChain("TruthTuple");
TruthChain.Add(TruthTupleFilename.c_str());

### Read all the momentum values from the MC and count entries in each bin

In [9]:
int NumberOfParticles;
int ParticleIDs[20];
double True_Px[20];
double True_Py[20];
double True_Pz[20];
TruthChain.SetBranchAddress("NumberOfParticles", &NumberOfParticles);
TruthChain.SetBranchAddress("ParticleIDs", ParticleIDs);
TruthChain.SetBranchAddress("True_Px", True_Px);
TruthChain.SetBranchAddress("True_Py", True_Py);
TruthChain.SetBranchAddress("True_Pz", True_Pz);

### Function to find index of kaons or pions

In [10]:
std::size_t FindIndex(int *ParticleIDs, int NumberOfParticles) {
    for(std::size_t i = 0; i < NumberOfParticles - 1; i++) {
        if(TMath::Abs(ParticleIDs[i]) == ID && TMath::Abs(ParticleIDs[i + 1]) == ID) {
            return i;
        }
    }
    std::cout << "Something went wrong!\n";
    return -1;
}

### Fill the histogram

In [11]:
for(std::size_t i = 0; i < TruthChain.GetEntries(); i++) {
    TruthChain.GetEntry(i);
    auto Index = FindIndex(ParticleIDs, NumberOfParticles);
    double Momentum = TMath::Sqrt(True_Px[Index]*True_Px[Index]
                                + True_Py[Index]*True_Py[Index]
                                + True_Pz[Index]*True_Pz[Index]);
    std::size_t Bin = FindBin(Momentum);
    Hist[Bin]++;
    Index++;
    Momentum = TMath::Sqrt(True_Px[Index]*True_Px[Index]
                         + True_Py[Index]*True_Py[Index]
                         + True_Pz[Index]*True_Pz[Index]);
    Bin = FindBin(Momentum);
    Hist[Bin]++;
}

### Function to do weighted average of efficiency differences

In [12]:
double GetWeightedAverage(const std::vector<double> &Numbers,
                          const std::vector<std::size_t> &Weights) {
    if(Numbers.size() != Weights.size()) {
        return 0.0;
    }
    double Sum = 0.0;
    double SumWeights = 0.0;
    for(std::size_t i = 0; i < Numbers.size(); i++) {
        Sum += Numbers[i]*Weights[i];
        SumWeights += Weights[i];
    }
    return Sum/SumWeights;
}

In [13]:
std::cout << "Average difference between data and MC: " << GetWeightedAverage(EffDiff, Hist) << " %\n";

Average difference between data and MC: -0.106921 %
